DD: delete
cmd + enter: new cell
ctrl + enter: run


In [1]:
# from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI, ChatOllama, ChatAnthropic
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

# llm = OpenAI(model_name="gpt-3.5-turbo")


chat = ChatOpenAI(
    temperature=0.1,  # how creative the model will be
    streaming=True,  # able to see model response progress
    callbacks=[StreamingStdOutCallbackHandler()],
)


template = PromptTemplate.from_template(
    "What is the distance between {country_a} and {country_b}.",
)

prompt = template.format(country_a="Mexico", country_b="Thailand")


# a = llm.predict("How many planets are there?")
# b = chat.predict("How many planets are there?")

chat.predict(prompt)

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_84010/2655004919.py:9: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI(
/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_84010/2655004919.py:26: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chat.predict(prompt)


The distance between Mexico and Thailand is approximately 16,000 kilometers (9,942 miles) when measured in a straight line.

'The distance between Mexico and Thailand is approximately 16,000 kilometers (9,942 miles) when measured in a straight line.'

In [2]:
from langchain.schema import HumanMessage, AIMessage, SystemMessage


messages = [
    SystemMessage(
        content="You are a geography expert. And you only reply in {language}"
    ),
    AIMessage(content="Ciao, mi chiamo {name}"),
    HumanMessage(
        content="What is the distance between {country_a} and {country_b}. Also, what is your name?"
    ),
]

chat.predict_messages(messages)

/var/folders/79/z024_lq9495450db95hryz680000gn/T/ipykernel_84010/2574164651.py:14: LangChainDeprecationWarning: The method `BaseChatModel.predict_messages` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chat.predict_messages(messages)


Mi dispiace, non posso calcolare la distanza tra due paesi senza informazioni specifiche sulla città di partenza e di arrivo. Il mio nome è Assistente Geografico. Come posso aiutarti oggi?

AIMessage(content='Mi dispiace, non posso calcolare la distanza tra due paesi senza informazioni specifiche sulla città di partenza e di arrivo. Il mio nome è Assistente Geografico. Come posso aiutarti oggi?', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-b55dbe45-28e2-4041-9b52-f3cccb4efa36-0')

In [3]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert. And you only reply in {language}"),
        ("ai", "Ciao, mi chiamo {name}"),
        (
            "human",
            "What is the distance between {country_a} and {country_b}. Also, what is your name?",
        ),
    ]
)

prompt = template.format_messages(
    language="Greek", name="Socrates", country_a="Mexico", country_b="Thailand"
)
chat.predict_messages(prompt)

Γεια σας! Το όνομά μου είναι Σωκράτης. Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 16.000 χιλιόμετρα.

AIMessage(content='Γεια σας! Το όνομά μου είναι Σωκράτης. Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 16.000 χιλιόμετρα.', additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-70f4281e-d62e-40b9-a82b-e71521d647e7-0')

In [4]:
from langchain.schema import BaseOutputParser


class CommaOutputParser(BaseOutputParser):

    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))


p = CommaOutputParser()

p.parse("Hello, how, are, you")

['Hello', 'how', 'are', 'you']

In [5]:
template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. DO NOT reply with anything else.",
        ),
        ("human", "{question}"),
    ]
)

prompt = template.format_messages(max_items=10, question="what are the colors?")

result = chat.predict_messages(prompt)

p = CommaOutputParser()
p.parse(result.content)

red, blue, green, yellow, orange, purple, pink, black, white, brown

['red',
 'blue',
 'green',
 'yellow',
 'orange',
 'purple',
 'pink',
 'black',
 'white',
 'brown']

In [6]:
chain = template | chat | CommaOutputParser()
chain.invoke({"max_items": 5, "question": "what are the pokemon?"})

pikachu, charmander, bulbasaur, squirtle, jigglypuff

['pikachu', 'charmander', 'bulbasaur', 'squirtle', 'jigglypuff']

In [7]:
# Chef chain
chef_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients.",
        ),
        ("human", "I want to cook {cuisine} food."),
    ]
)

chef_chain = chef_prompt | chat

In [8]:
# Vegetarian Chef chain
veg_chef_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a vegetarian chef specialized on making traditional recipies vegetarian. You   find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it.",
        ),
        ("human", "{recipe}"),
    ]
)

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe": chef_chain} | veg_chain  # LCEL (LangChain Expression Language)

In [9]:
final_chain.invoke({"cuisine": "indian"})

Great choice! Indian cuisine is known for its bold flavors and aromatic spices. Let's start with a classic and popular dish - Chicken Tikka Masala. Here's a simple recipe for you to try at home:

Chicken Tikka Masala

Ingredients:
- 1 lb boneless, skinless chicken breasts, cut into bite-sized pieces
- 1 cup plain yogurt
- 2 tablespoons lemon juice
- 2 teaspoons ground cumin
- 2 teaspoons ground coriander
- 1 teaspoon ground turmeric
- 1 teaspoon chili powder
- 1 teaspoon paprika
- 1 teaspoon garam masala
- 2 cloves garlic, minced
- 1-inch piece of ginger, grated
- Salt and pepper to taste
- 2 tablespoons vegetable oil
- 1 onion, finely chopped
- 1 can (14 oz) tomato sauce
- 1 cup heavy cream
- Fresh cilantro, chopped (for garnish)

Instructions:
1. In a bowl, combine the yogurt, lemon juice, cumin, coriander, turmeric, chili powder, paprika, garam masala, garlic, ginger, salt, and pepper. Add the chicken pieces and mix well to coat. Cover and marinate in the refrigerator for at least 1

AIMessage(content="For a vegetarian version of Chicken Tikka Masala, you can replace the chicken with a plant-based alternative such as tofu or paneer. Here's how you can prepare the tofu or paneer as a substitute for the chicken in this recipe:\n\n**Tofu:**\n1. Use firm or extra-firm tofu for this recipe. Press the tofu to remove excess water by wrapping it in a clean kitchen towel and placing a heavy object on top for about 15-20 minutes.\n2. Cut the tofu into bite-sized cubes and follow the marinade instructions in the recipe. Tofu absorbs flavors well, so marinating it for a longer time can enhance the taste.\n3. Instead of baking, you can pan-fry the marinated tofu in a little oil until it's golden brown and slightly crispy on the outside.\n\n**Paneer:**\n1. Paneer is a type of Indian cheese that holds its shape well when cooked. You can find it in Indian grocery stores or make it at home by curdling milk with lemon juice or vinegar.\n2. Cut the paneer into cubes and follow the ma